# Week 12 guided lab · A high-pollution day classifier, step by step

**CMT124** · two-hour **guided** lab

**The question:** can we warn of high-pollution days from tomorrow's temperature and traffic, which mistakes will such a warning make, and would it still work somewhere warmer?

You'll start from where Week 11 ended (the cleaned air-quality data and the two features that feature selection kept) and build a classifier the way it should be built: define the class without leaking test information, check a baseline, read the mistakes, choose a threshold, and test the model on data from somewhere else.

### How this guided lab works

As in Week 11, **we work through it together, one step at a time.**

| Each step has | What you do |
|---|---|
| **Do** | one operation, in the empty code cell |
| **Checkpoint** | run the checkpoint cell and compare what it prints with your neighbours |
| **Think** | a question we answer together; jot down the answer in the notes cell |

* Only one step is open at a time. Finished early? Read the next **Think** question and prepare an answer.
* A checkpoint that prints `✔` means your result matches. An `AssertionError` tells you what to look at.
* Stuck? Under the key steps there is a cell headed *"If your result does not match the checkpoint, run this cell to continue."*
* Everything uses Week 11 and today's lectures: Boolean filtering, `train_test_split`, `quantile`, `RandomForestClassifier`, `predict` / `predict_proba`, `confusion_matrix`, precision / recall / F1, `joblib`. **No Earth Engine is needed.**

| Step | One operation | Checkpoint |
|---|---|---|
| 1 | rebuild the cleaned data and split it | training and test shapes |
| 2 | define the high-pollution class from the training days | the threshold and the class counts |
| 3 | the "always normal" baseline | its accuracy |
| 4 | fit a random forest classifier | its accuracy |
| 5 | the confusion matrix | four counts |
| 6 | precision, recall and F1 | three scores |
| 7 | probabilities and the decision threshold | a table of three thresholds |
| 8 | a year of data from a warmer city | the same scores, somewhere new |
| 9 | save, reload and use the model | identical predictions |

### Setup

Run this cell first. It loads the libraries, the Week 11 generator `make_air_quality()`, and a helper `class_scores(y_true, y_pred)` that returns accuracy, precision, recall and F1 of the high-pollution class (we'll use it from Step 7, once you've computed the scores yourself).

*The data are synthetic: daily records made by a formula plus random noise for teaching, not measurements from a real site. The high-pollution class is a teaching definition, not an official air-quality threshold.*

In [ ]:
#@title Setup: run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)


def make_air_quality(n=5000, seed=3622626236, temp_mean=20):
    """Synthetic daily records at one roadside site: weather, traffic and PM2.5 (µg/m³)."""
    rng = np.random.RandomState(seed)
    temperature = rng.normal(temp_mean, 5, n)            # °C
    humidity = rng.uniform(30, 90, n)                    # %
    wind_speed = rng.exponential(2, n)                   # m/s
    traffic_intensity = rng.uniform(0, 100, n)           # arbitrary units
    pm25 = (5 + 0.4 * traffic_intensity
            + 0.1 * traffic_intensity * (temperature - 20)
            + 0.3 * (30 - np.clip(wind_speed, 0, 30))
            + 0.1 * (humidity - 50)
            - 0.3 * (temperature - 20))
    noise = 0.5 * rng.normal(0, 10, n) + 0.5 * rng.normal(5, 5, n)
    return pd.DataFrame({"temperature": temperature, "humidity": humidity,
                         "wind_speed": wind_speed, "traffic_intensity": traffic_intensity,
                         "pm25": pm25 + noise})


def class_scores(y_true, y_pred):
    """Accuracy, and precision / recall / F1 of class 1 (high-pollution days)."""
    return {"accuracy": round(accuracy_score(y_true, y_pred), 3),
            "precision": round(precision_score(y_true, y_pred), 3),
            "recall": round(recall_score(y_true, y_pred), 3),
            "F1": round(f1_score(y_true, y_pred), 3)}


air = make_air_quality()
print(air.shape)

## Step 1 · Rebuild the cleaned data and split it

**Step 1 of 9** · ●○○○○○○○○

Exactly the Week 11 starting point, with the two selected features.

**Do:**
1. `air_clean` = the rows of `air` with `pm25` at least zero;
2. `X` = its columns `["temperature", "traffic_intensity"]`; `pm25` = its `pm25` column;
3. split into `X_train, X_test, pm25_train, pm25_test` with `test_size=0.2` and `random_state=42`.

In [ ]:
# STUDENT TASK
# Task: air_clean (pm25 >= 0), X (two features), pm25 (target values), then an 80/20 split with random_state=42
# Hint: X_train, X_test, pm25_train, pm25_test = train_test_split(X, pm25, test_size=0.2, random_state=42)
# your code here


### Checkpoint 1

In [ ]:
# Checkpoint 1: the same split as in Week 11
print("training:", X_train.shape, "  test:", X_test.shape)
assert X_train.shape == (3787, 2) and X_test.shape == (947, 2), "expected (3787, 2) and (947, 2): check the cleaning and the two columns"
print("✔ Step 1")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
air_clean = air[air["pm25"] >= 0]
X = air_clean[["temperature", "traffic_intensity"]]
pm25 = air_clean["pm25"]
X_train, X_test, pm25_train, pm25_test = train_test_split(X, pm25, test_size=0.2, random_state=42)

### Think 1

In Week 11, `y` was the PM2.5 value. Today we split the **PM2.5 values first** and only then turn them into classes. Why split first?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 2 opens.

## Step 2 · Define the high-pollution class from the training days

**Step 2 of 9** · ●●○○○○○○○

Our teaching definition: a **high-pollution day** has PM2.5 above the **90th percentile of the training days**.

**Do:**
1. `threshold` = the 0.9 quantile of `pm25_train`;
2. `high_train` = 1 where `pm25_train` is above `threshold`, else 0 (integers);
3. `high_test` = the same rule, with the **same** `threshold`, for `pm25_test`.

In [ ]:
# STUDENT TASK
# Task: threshold from the TRAINING values only; high_train and high_test as 0/1 integers
# Hint: threshold = pm25_train.quantile(0.9); high_train = (pm25_train > threshold).astype(int)
# your code here


### Checkpoint 2

In [ ]:
# Checkpoint 2: the threshold and the class counts
print(f"threshold: {threshold:.2f} µg/m³")
print("high days: training", high_train.sum(), f"({high_train.mean():.1%})", "  test", high_test.sum(), f"({high_test.mean():.1%})")
print("for comparison, the 0.9 quantile of ALL cleaned days:", round(pm25.quantile(0.9), 2))
assert abs(threshold - 79.42) < 0.01, "use the 0.9 quantile of pm25_train"
assert (high_train.sum(), high_test.sum()) == (379, 101), "same threshold for both sets; > threshold means high"
print("✔ Step 2")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
threshold = pm25_train.quantile(0.9)
high_train = (pm25_train > threshold).astype(int)
high_test = (pm25_test > threshold).astype(int)

### Think 2

Exactly 10.0 % of the training days are "high", but 10.7 % of the test days. Why aren't both exactly 10 %? And what would you need to change to turn this into a classifier that a real warning service could use?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 3 opens.

## Step 3 · The "always normal" baseline

**Step 3 of 9** · ●●●○○○○○○

**Do:** make `baseline_pred`, an array of zeros with one entry per test day (a "model" that always says *normal*), and compute its accuracy on `high_test` as `baseline_acc`.

In [ ]:
# STUDENT TASK
# Task: baseline_pred = all zeros (one per test day); baseline_acc = its accuracy on high_test
# Hint: np.zeros(len(high_test), dtype=int); accuracy_score(high_test, baseline_pred)
# your code here


### Checkpoint 3

In [ ]:
# Checkpoint 3: the accuracy of saying "normal" every day
print("baseline accuracy:", round(baseline_acc, 3))
print("high-pollution days it finds:", int((baseline_pred[high_test.to_numpy() == 1] == 1).sum()), "of", high_test.sum())
assert abs(baseline_acc - 0.893) < 0.001, "all zeros, compared with high_test"
print("✔ Step 3")

### Think 3

A model that has never found a single high-pollution day is **89 % accurate**. Is accuracy enough to judge a warning system? What would you look at instead?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 4 opens.

## Step 4 · Fit a random forest classifier

**Step 4 of 9** · ●●●●○○○○○

**Do:** fit a `RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)` called `forest` on `X_train` and `high_train`, then predict the test days as `pred`.

In [ ]:
# STUDENT TASK
# Task: forest = RandomForestClassifier (200 trees, random_state=42) fitted to high_train; pred = predicted classes of the test days
# Hint: the same create → fit → predict as in Week 11, with the class as target: forest.fit(X_train, high_train)
# your code here


### Checkpoint 4

In [ ]:
# Checkpoint 4: accuracy, and how many days it calls "high"
print("forest accuracy:", round(accuracy_score(high_test, pred), 3), "  (baseline:", round(baseline_acc, 3), ")")
print("days predicted high:", pred.sum(), "  days actually high:", high_test.sum())
assert abs(accuracy_score(high_test, pred) - 0.967) < 0.003, "fit on X_train and high_train (not pm25_train), predict X_test"
print("✔ Step 4")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
forest.fit(X_train, high_train)
pred = forest.predict(X_test)

### Think 4

The forest scores **96.7 %** against the baseline's 89.3 %. It predicts 90 days as high, and there are 101. What can you already say about its mistakes, and what can't you say yet?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 5 opens.

## Step 5 · The confusion matrix

**Step 5 of 9** · ●●●●●○○○○

**Do:** compute the confusion matrix of the forest's test predictions as `cm`, then draw it with `ConfusionMatrixDisplay` with the labels `"normal"` and `"high"`.

In [ ]:
# STUDENT TASK
# Task: cm = confusion_matrix of the test classes and pred; draw it with the labels normal / high
# Hint: ConfusionMatrixDisplay(cm, display_labels=["normal", "high"]).plot(cmap="Blues", colorbar=False)
# your code here


### Checkpoint 5

In [ ]:
# Checkpoint 5: the four counts (rows = actual, columns = predicted)
tn, fp, fn, tp = cm.ravel()
print(f"true negatives {tn} · false positives (false alarms) {fp} · false negatives (missed days) {fn} · true positives {tp}")
assert (tn, fp, fn, tp) == (836, 10, 21, 80), "confusion_matrix(actual, predicted): actual first"
print("✔ Step 5")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
cm = confusion_matrix(high_test, pred)
tn, fp, fn, tp = cm.ravel()

### Think 5

The forest raised **10 false alarms** and **missed 21 high-pollution days**. For a public health warning, which of the two mistakes is more costly, and for whom? Would a forecast used to **close a road** see it differently?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 6 opens.

## Step 6 · Precision, recall and F1

**Step 6 of 9** · ●●●●●●○○○

**Do:** compute `precision`, `recall` and `f1` of the high-pollution class with `precision_score`, `recall_score` and `f1_score` (each takes the actual classes first, then the predictions).

In [ ]:
# STUDENT TASK
# Task: precision, recall and f1 of the forest on the test days
# Hint: precision = precision_score(high_test, pred), and the same pattern for recall_score and f1_score
# your code here


### Checkpoint 6

In [ ]:
# Checkpoint 6: the scores, and the same scores by hand from the counts of Step 5
print(f"precision {precision:.3f} = {tp}/({tp}+{fp}) = {tp / (tp + fp):.3f}")
print(f"recall    {recall:.3f} = {tp}/({tp}+{fn}) = {tp / (tp + fn):.3f}")
print(f"F1        {f1:.3f}")
assert abs(precision - 0.889) < 0.002 and abs(recall - 0.792) < 0.002, "actual classes first: precision_score(high_test, pred)"
print("✔ Step 6")

### Think 6

**Halfway question.** Put precision **0.89** and recall **0.79** into one sentence each that a member of the public would understand. Then: what have we learned so far that accuracy alone would have hidden?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 7 opens.

## Step 7 · Probabilities and the decision threshold

**Step 7 of 9** · ●●●●●●●○○

`predict` calls a day "high" when the forest's probability is at least 0.5. We can choose a different threshold.

**Do:** store the forest's probabilities of a high-pollution day for the test days as `p_high` (column 1 of `predict_proba`). The checkpoint then classifies the test days at three thresholds.

In [ ]:
# STUDENT TASK
# Task: p_high = probability of class 1 (high) for every test day
# Hint: forest.predict_proba(X_test) has one column per class; take [:, 1]
# your code here


### Checkpoint 7

In [ ]:
# Checkpoint 7: three decision thresholds, one table
rows = {}
for t in [0.3, 0.5, 0.7]:
    pred_t = (p_high >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(high_test, pred_t).ravel()
    rows[f"threshold {t}"] = {**class_scores(high_test, pred_t), "false alarms": fp_t, "missed days": fn_t}
print(pd.DataFrame(rows).T.astype({"false alarms": int, "missed days": int}))
assert len(p_high) == 947 and p_high.min() >= 0 and p_high.max() <= 1, "p_high: one probability per test day, column 1"
assert (p_high >= 0.5).astype(int).sum() == pred.sum(), "at threshold 0.5 you should get the forest's own predictions"
print("✔ Step 7")

### Think 7

Which threshold would you choose for a **public health warning**, and what does that choice cost? Who should make this decision: the data scientist or the service that issues the warnings?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 8 opens.

## Step 8 · A year of data from a warmer city

**Step 8 of 9** · ●●●●●●●●○

The real test of a model is data it has never seen, **from somewhere else**. `make_air_quality(n=365, seed=2027, temp_mean=24)` simulates one year of daily records from a city that is on average **4 °C warmer** than the one we trained on.

**Do:**
1. make `city` with that call, and clean it **the same way** (keep `pm25` ≥ 0) as `city_clean`;
2. `high_city` = its classes using our **training** `threshold` (not a new one);
3. `pred_city` = the forest's predictions for its `["temperature", "traffic_intensity"]` columns.

In [ ]:
# STUDENT TASK
# Task: city_clean (365 simulated days, cleaned as before), high_city (training threshold), pred_city (forest predictions)
# Hint: high_city = (city_clean["pm25"] > threshold).astype(int); forest.predict(city_clean[["temperature", "traffic_intensity"]])
# your code here


### Checkpoint 8

In [ ]:
# Checkpoint 8: the same scores, in the new city
print("days after cleaning:", len(city_clean), " high-pollution days:", high_city.sum(), f"({high_city.mean():.1%})")
print("hottest day: training", round(X_train["temperature"].max(), 1), "°C · new city", round(city_clean["temperature"].max(), 1), "°C")
print(pd.DataFrame({"test days (training city)": class_scores(high_test, pred),
                    "new city": class_scores(high_city, pred_city)}).T)
print(confusion_matrix(high_city, pred_city))
assert len(city_clean) == 360 and high_city.sum() == 79, "n=365, seed=2027, temp_mean=24; remove pm25 < 0; use the training threshold"
print("✔ Step 8")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
city = make_air_quality(n=365, seed=2027, temp_mean=24)
city_clean = city[city["pm25"] >= 0]
high_city = (city_clean["pm25"] > threshold).astype(int)
pred_city = forest.predict(city_clean[["temperature", "traffic_intensity"]])

### Think 8

In the warmer city, **22 %** of days are "high" by our training threshold, instead of 10 %, yet precision and recall stay high. Why does the model transfer so well **here**? Would you expect the same from a **real** second city? And is "high" still a sensible label when one day in five gets it?

*Notes from the discussion:*

**Pause.** We compare checkpoints and answer the question together before Step 9 opens.

## Step 9 · Save, reload and use the model

**Step 9 of 9** · ●●●●●●●●●

A model in use is trained once, saved, and loaded wherever predictions are needed.

**Do:** save `forest` to the file `"high_pm25_forest.joblib"` with `dump`, load it back as `loaded`, and predict the new city's days with it as `pred_loaded`.

In [ ]:
# STUDENT TASK
# Task: dump forest to "high_pm25_forest.joblib", load it as loaded, pred_loaded = its predictions for the new city
# Hint: from joblib import dump, load
# your code here


### Checkpoint 9

In [ ]:
# Checkpoint 9: the loaded model must give exactly the same predictions
print("identical predictions:", (pred_loaded == pred_city).all())
assert (pred_loaded == pred_city).all(), "load the file you saved and predict city_clean[['temperature', 'traffic_intensity']]"
print("✔ Step 9")

### Think 9

The saved file contains the forest, but not everything the forest **depends on**. List what else a warning service would have to store and keep identical, and what you would **monitor** once it runs every day.

*Notes from the discussion:*

## Optional: if you finish early

None of this is needed after the course.

**A · Logistic regression and AUC.** Fit `LogisticRegression(max_iter=1000)` (from `sklearn.linear_model`) on the same training data and compare its `class_scores` and its ROC AUC (`roc_auc_score(high_test, probabilities)` from `sklearn.metrics`) with the forest's. Which would you choose, and why?

**B · Weighting the rare class.** Refit the forest with `class_weight="balanced"`. What happens to recall and precision? Is that different from lowering the threshold?

**C · Real satellite data.** `W12_landcover_rf_student.ipynb` runs the same workflow on 3,600 labelled Sentinel-2 pixels (three land-cover classes). Its model part works without Earth Engine; with Earth Engine access you can also rebuild the pixel table yourself.

**D · Deep learning.** `W12_deep_learning_optional.ipynb` shows what changes when a neural network learns its own features from images.

In [ ]:
# your code here

## Wrap-up

* Define the class from the **training** data and say what it means: ours is a teaching definition, not an official threshold.
* Beat a **baseline**, then read the **mistakes**: confusion matrix, precision, recall, F1.
* The **decision threshold** is a choice made for the users of the model.
* Test on data from **somewhere else**, store everything the model depends on, and **monitor** it in use.

That's the full workflow of the module: **clean → split → fit → evaluate → interpret → question**.